# Lesson 12 - Dates, Decimal, and Rounding

**Goal:** Handle calendar values and monetary arithmetic with explicit precision and rounding rules.

**Level:** intermediate. Complete the preceding chapters first. This notebook defines its own
inputs and runs independently. Run cells top to bottom with **Shift+Enter**. Predict results,
inspect the output, and change one input at a time. No external services or third-party packages
are required beyond your Jupyter setup.

Examples include lightweight executable checks. Practice cells intentionally contain no solutions;
expected results help you verify your own work. Temporary files and databases are isolated and cleaned up.

Where salary examples appear, use the course's simplified exercise rules: DA 48% of basic,
HRA 16% capped at 16,000 monthly, PF 12% of basic plus DA, and annual tax 20% only on annual
gross above 12,00,000, divided by 12 for monthly tax. PF is a separate deduction.
These are learning assumptions, not actual tax or payroll rules.

## 1. Parse and format dates

Use date objects for calendar values. ISO text YYYY-MM-DD is convenient for storage. strftime formats a date for display. Invalid dates raise ValueError; do not compare arbitrary formatted date strings chronologically.

In [1]:
from datetime import date, timedelta

joining_date = date.fromisoformat("2024-04-01")
review_date = date.fromisoformat("2026-04-01")
print(joining_date.strftime("%d %b %Y"))
print("Elapsed days:", (review_date - joining_date).days)
print("Next day:", review_date + timedelta(days=1))
assert (review_date - joining_date).days == 730

01 Apr 2024
Elapsed days: 730
Next day: 2026-04-02


## 2. Calendar boundaries

A month is not always 30 days. Use calendar.monthrange for its actual length. Define a policy for anniversaries on February 29 rather than silently guessing.

In [2]:
import calendar

for year, month in [(2024, 2), (2025, 2), (2026, 4)]:
    days = calendar.monthrange(year, month)[1]
    print(year, month, days)
assert calendar.monthrange(2024, 2)[1] == 29

2024 2 29
2025 2 28
2026 4 30


## 3. Decimal from strings

Binary floats cannot represent every decimal fraction exactly. Construct Decimal from strings or integers, not from a float when exact decimal input matters. Do not mix Decimal and float in arithmetic.

In [3]:
from decimal import Decimal, ROUND_HALF_UP

print(0.1 + 0.2 == 0.3)
exact = Decimal("0.1") + Decimal("0.2")
print(exact, exact == Decimal("0.3"))
amount = Decimal("100.005")
rounded = amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print(rounded)
assert rounded == Decimal("100.01")

False
0.3 True
100.01


## 4. Make the rounding contract explicit

This example calculates using Decimal and rounds each deduction to cents before subtraction. A different policy can change totals. Record which stage is rounded. The tax threshold remains based on annual gross, not on gross minus PF.

In [4]:
def money(value):
    return value.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

basic = Decimal("112600.00")
da = money(basic * Decimal("0.48"))
hra = min(money(basic * Decimal("0.16")), Decimal("16000.00"))
gross = basic + da + hra + Decimal("1000.00")
pf = money((basic + da) * Decimal("0.12"))
annual_tax = money(max(Decimal("0"), gross * 12 - Decimal("1200000")) * Decimal("0.20"))
monthly_tax = money(annual_tax / 12)
net = gross - pf - monthly_tax
print(gross, pf, monthly_tax, net)
assert net == Decimal("146920.64")

183648.00 19997.76 16729.60 146920.64


## Practice lab

Define all inputs in each answer cell, use `practice_` names where helpful, and explain important
decisions in comments. Try a task before looking at its expected results. Add at least one normal,
one boundary, and one invalid-input check where the task has an input contract.
Keep file exercises inside TemporaryDirectory until you deliberately choose a permanent output path.

### Exercise 1. Parse dates

Parse 2026-04-01 and 2026-04-15 and calculate elapsed days.

**Check:** 14 elapsed days; decide separately if a business rule counts both endpoints.

In [5]:
# Write your solution and checks here.

### Exercise 2. Validate a calendar date

Attempt to parse 2025-02-29 and catch ValueError; then parse 2024-02-29.

**Check:** Only the leap-year date is valid.

In [6]:
# Write your solution and checks here.

### Exercise 3. Choose a rounding policy

Round Decimal("2.345") and Decimal("2.355") to cents using ROUND_HALF_UP and ROUND_HALF_EVEN.

**Check:** Half-up gives 2.35 and 2.36; half-even gives 2.34 and 2.36.

In [7]:
# Write your solution and checks here.

### Exercise 4. Decimal salary

Calculate the 75,000 basic, 1,500 electricity case entirely with Decimal.

**Check:** Gross 124500.00, PF 13320.00, tax 4900.00, net 106280.00.

In [8]:
# Write your solution and checks here.

### Exercise 5. Serialize money

Represent Decimal amounts as strings in JSON, then reconstruct them on reading.

**Check:** Decimal("100.10") round-trips without converting through float.

In [9]:
# Write your solution and checks here.

## Chapter project

Add joining dates and Decimal salaries to three employee records. Produce a report for a fixed date, document rounding, and avoid dependence on the computer clock.

In [10]:
# Build your chapter project here. Separate inputs, logic, and display.

## Reflection

1. Which input assumptions does your solution enforce, and which does it document?
2. What happens for empty input, a boundary value, or a malformed record?
3. Which parts can be reused without the notebook?
4. What would make the code clearer to another learner?

Restart the kernel and run all cells after completing your work to catch hidden dependencies.

**Next:** `13_classes_and_composition.ipynb` - Classes and Composition.